In [5]:
import warnings
import pandas as pd
import os
import logging
from datetime import datetime

In [2]:
def handle_multiple_orders(price_data, signal_datetime, side, num_orders, pct_orders, pct_order_range, entry_time_offset, percentage_change, open_order_elimination, tp_first_order, tp_second_order, sl):
    orders = []
    total_margin_used = 0
    total_invested = 0
  
    intended_total_margin_used = 0
    first_tp_datetime = None
    second_order_entry_datetime = None
    first_order_datetime = None
    entry_price = None
    tp_price_first_order = None

    for i in range(num_orders):
        margin_percentage = pct_orders[i]
        intended_total_margin_used += margin_percentage

        if intended_total_margin_used > 1.0:
            raise ValueError("Total margin percentage exceeds 100%")

        if i == 0:
            # First order
            current_signal_datetime = signal_datetime
            entry_datetime, entry_price, entry_duration = determine_entry(
                price_data, current_signal_datetime, percentage_change, side, open_order_elimination, entry_time_offset
            )

            if entry_datetime is None:
                return None, None, None, None, None, None, None, None

            first_order_datetime = entry_datetime
            orders.append((entry_datetime, entry_price, margin_percentage, entry_duration))
            total_invested += margin_percentage * entry_price
            total_margin_used += margin_percentage

            # Calculate TP for the first order
            tp_price_first_order = entry_price * (1 + tp_first_order if side == 'Buy' else 1 - tp_first_order)

            # Find the datetime when the TP or SL is hit for the first order
            for price_time, price_row in price_data.loc[entry_datetime:].iterrows():
                if side == 'Buy':
                    if first_tp_datetime is None and price_row['High'] >= tp_price_first_order:
                        first_tp_datetime = price_time
                else:
                    if first_tp_datetime is None and price_row['Low'] <= tp_price_first_order:
                        first_tp_datetime = price_time

                if first_tp_datetime:
                    break

        else:
            # Calculate the entry price for the second order based on the first order's price and pct_order_range
            entry_price_offset = pct_order_range * (1 if side == 'Sell' else -1)
            previous_order_price = orders[-1][1]
            adjusted_entry_price = previous_order_price * (1 + entry_price_offset)

            subsequent_prices = price_data.loc[orders[-1][0]:]
            valid_order = False

            for price_time, price_row in subsequent_prices.iterrows():
                if (side == 'Buy' and price_row['Low'] <= adjusted_entry_price) or (side == 'Sell' and price_row['High'] >= adjusted_entry_price):
                    valid_order = True
                    entry_datetime = price_time
                    entry_price = adjusted_entry_price
                    entry_duration = (entry_datetime - signal_datetime).total_seconds() / 60  # Duration in minutes
                    second_order_entry_datetime = entry_datetime
                    break

            if valid_order:
                total_invested += margin_percentage * entry_price
                total_margin_used += margin_percentage
                orders.append((entry_datetime, entry_price, margin_percentage, entry_duration))
            else:
                intended_total_margin_used -= margin_percentage  # Adjust intended margin used if the order was not placed

    if not orders:
        return None, None, None, None, None, None, None, None

    # Calculate the intended average entry price across all orders placed
    intended_average_entry_price = total_invested / total_margin_used if total_margin_used > 0 else None

    # Calculate SL based on the intended average entry price
    sl_price = intended_average_entry_price * (1 - sl if side == 'Buy' else 1 + sl) if intended_average_entry_price else None

    # Calculate TP for the second order (if placed)
    tp_price_second_order = intended_average_entry_price * (1 + tp_second_order if side == 'Buy' else 1 - tp_second_order) if intended_average_entry_price else None

    # Compare the second order's entry datetime with the first order's TP datetime
    filled_orders = 2 if second_order_entry_datetime and (not first_tp_datetime or second_order_entry_datetime <= first_tp_datetime) else 1

    return orders, intended_average_entry_price, filled_orders, first_order_datetime, entry_price, tp_price_first_order, tp_price_second_order, sl_price


In [6]:
# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def backtest_trades(price_data, signal_data, num_orders=None, pct_orders=None, pct_order_range=None, tp_first_order=None, tp_second_order=None, sl=None, entry_time_offset=None, percentage_change=None, open_order_elimination=None, ignore_time_interval_before=None, ignore_time_interval_after=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'First Order Entry Price', ' Average Entry Price', 'Second Order Entry Price', 'TP Price First Order', 'TP Price Second Order', 'SL Price', 'Result', 
        'Exit Datetime', 'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])

    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []
    initial_drawdown = 0  # For initial drawdown calculation
    nav_history = []
    
    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']

        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'

        # Ignoring signals based on open trades
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''

        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 2:
                result = 'Ignored'
                reason = '2 open trades'
                ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'First Order Entry Price': None,
                'Average Entry Price': None,
                'Second Order Entry Price': None,
                'TP Price First Order': None,
                'TP Price Second Order' : None,
                'SL Price': None,
                'Result': result,
                'Exit Datetime': exit_datetime,
                'Execution Latency' : None,
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        # New logic: ignore signals within a specific time interval before and after events
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            ignore_signal = any(event_datetime - pd.Timedelta(
                minutes=ignore_time_interval_before) <= signal_datetime <= event_datetime + pd.Timedelta(
                minutes=ignore_time_interval_after)
                                for event_datetime in event_times)
        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'First Order Entry Price': None,
                'Average Entry Price': None,
                'Second Order Entry Price': None,
                'TP Price First Order': None,
                'TP Price Second Order' : None,
                'SL Price': None,
                'Result': 'Ignored',
                'Exit Datetime': exit_datetime,
                'Execution Latency' : None,
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': 'Signal around economic event'
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, open_order_elimination, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'First Order Entry Price': None,
                'Average Entry Price': None,
                'Second Order Entry Price': None,
                'TP Price First Order': None,
                'TP Price Second Order' : None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Exit Datetime': exit_datetime,
                'Execution Latency' : None,
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        # Call the updated handle_multiple_orders function
        orders, average_entry_price, filled_orders, first_order_datetime, first_order_entry_price, tp_price_first_order, tp_price_second_order, sl_price = handle_multiple_orders(
            price_data, signal_datetime, side, num_orders, pct_orders, pct_order_range, entry_time_offset, percentage_change, open_order_elimination, tp_first_order ,tp_second_order, sl
        )

        if not orders:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'First Order Entry Price': None,
                'Average Entry Price': None,
                'Second Order Entry Price': None,
                'TP Price First Order': None,
                'TP Price Second Order' : None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Exit Datetime': exit_datetime,
                'Execution Latency' : None,
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        current_margin *= (1 - 0.0002 * filled_orders)  # Reduce margin by commission for each order filled
    
        # Determine which TP to use
        tp = tp_first_order if filled_orders == 1 else tp_second_order
        tp_price = tp_price_first_order if filled_orders == 1 else tp_price_second_order

        result, duration_str, exit_datetime = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = orders[-1][0] + pd.Timedelta(duration_str) if filled_orders == 2 else orders[0][0] + pd.Timedelta(duration_str)
        
        
        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            for event_datetime in event_times:
                if entry_datetime < event_datetime < exit_datetime:
                    exit_datetime = event_datetime - pd.Timedelta(minutes=10)
                    if exit_datetime in price_data.index:
                        exit_price = price_data.at[exit_datetime, 'Open']
                        result = 'ended before data'
                        if (side == 'Buy' and exit_price > entry_price) or (
                                side == 'Sell' and exit_price < entry_price):
                            result += ' with profit'
                            pct_change = (exit_price - entry_price) / entry_price if side == 'Buy' else (
                                                                                                                entry_price - exit_price) / entry_price
                            current_margin = current_margin * (1 + pct_change)
                            current_margin *= (1 - 0.0005)
                        else:
                            result += ' with loss'
                            pct_change = (entry_price - exit_price) / entry_price if side == 'Buy' else (
                                                                                                                exit_price - entry_price) / entry_price
                            current_margin = current_margin * (1 - pct_change)
                            current_margin *= (1 - 0.0005)
                    break

        if result not in ['ended before data with profit', 'ended before data with loss',
                          'ended before data with no exact price']:

            if result == 1:
                current_margin = current_margin * (1 + tp)
                current_margin *= (1 - 0.0005)
            elif result == -1:
                current_margin = current_margin * (1 - sl)
                current_margin *= (1 - 0.0005)

        # Calculate initial drawdown
        if current_margin < 100000:
            drawdown = ((100000 - current_margin) / 100000) * 100
            initial_drawdown = max(initial_drawdown, drawdown)

        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        if filled_orders==2:
            average_entry_price=average_entry_price
        else: 
            average_entry_price=  entry_price
        
        nav_history.append(nav)
        new_row = pd.DataFrame([{
        'Datetime': signal_datetime,
        'Side': side,
        'First Order Entry Price': entry_price,
        'Average Entry Price': average_entry_price,
        'Second Order Entry Price': orders[1][1] if len(orders) > 1 else None,
        'TP Price First Order': average_entry_price * (1 + tp_first_order if side == 'Buy' else 1 - tp_first_order),
        'TP Price Second Order': tp_price,  # This was calculated earlier
        'SL Price': sl_price,
        'Result': result,
        'Exit Datetime': exit_datetime,
        'Execution Latency': format_duration(entry_duration),
        'ROI': roi,
        'NAV': nav,
        'Ignore Reason': ''
         }])

        output_data = pd.concat([output_data, new_row], ignore_index=True)

    # Ensure 'Datetime' column is in datetime format
    output_data['Datetime'] = pd.to_datetime(output_data['Datetime'])

    # Calculate Daily Return
    output_data['Date'] = output_data['Datetime'].dt.date
    daily_nav = output_data.groupby('Date')['NAV'].last().to_dict()
    daily_returns = {}
    previous_day_nav = 100000

    for date, nav in daily_nav.items():
        daily_return = ((nav - previous_day_nav) / previous_day_nav) * 100
        daily_returns[date] = daily_return
        previous_day_nav = nav

    output_data['Daily Return'] = output_data['Date'].map(daily_returns)
    output_data.drop(columns=['Date'], inplace=True)

    # Calculate Monthly Max Drawdown
    output_data['Month'] = output_data['Datetime'].dt.to_period('M')
    monthly_max_drawdowns = {}

    for month, group in output_data.groupby('Month'):
        peak_nav = group['NAV'].iloc[0]  # Start with the first NAV of the month
        max_drawdown_in_month = 0
        local_peak = peak_nav

        for nav in group['NAV']:
            if nav > local_peak:
                local_peak = nav  # Update the peak if a new high is found
            else:
                # Calculate drawdown from the peak to the current NAV
                drawdown = ((local_peak - nav) / local_peak) * 100
                max_drawdown_in_month = max(max_drawdown_in_month, drawdown)  # Track the maximum drawdown

        monthly_max_drawdowns[month] = max_drawdown_in_month

    output_data['Monthly Max Drawdown'] = output_data['Month'].map(monthly_max_drawdowns)
    output_data['Initial Drawdown'] = initial_drawdown

    output_data.drop(columns=['Month'], inplace=True)

    return output_data
# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str, exit_datetime

def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

In [4]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\price_2023-06-01_to_2024-08-4_min.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\\filtered_signals_with_2023_2024_all_events_plus_one_hour.csv', parse_dates=['Datetime'])
# Define the month and year for which you want to perform the trades
month = 7  # January (you can change this to the desired month)
year = 2024  # You can change this to the desired year

# Filter the signal data for the specified month and year
signal_data = signal_data[(signal_data['Datetime'].dt.month == month) & (signal_data['Datetime'].dt.year == year)]


In [5]:
num_orders = 2
pct_orders = [0.5, 0.5]
pct_order_range = 0.01
tp_first_order = 0.01  
tp_second_order = 0.012
sl = 0.01
entry_time_offset = 0  # No time offset
percentage_change = 0  # 0.5% change for filling the order
open_order_elimination = 120
ignore_time_interval_before = 1020
ignore_time_interval_after = 0
output = backtest_trades(
    price_data=price_data,
    signal_data=signal_data,
    num_orders=num_orders,
    pct_orders=pct_orders,
    pct_order_range=pct_order_range,
    tp_first_order=tp_first_order,
    tp_second_order=tp_second_order,
    sl=sl,
    entry_time_offset=entry_time_offset,
    percentage_change=percentage_change,
    open_order_elimination=open_order_elimination,
    ignore_time_interval_before=ignore_time_interval_before,
    ignore_time_interval_after=ignore_time_interval_after
)
output

,Datetime,Side,First Order Entry Price,Average Entry Price,Second Order Entry Price,TP Price First Order,TP Price Second Order,SL Price,Result,Exit Datetime,Execution Latency,ROI,NAV,Ignore Reason,Average Entry Price,Daily Return,Monthly Max Drawdown,Initial Drawdown
0,2024-07-02 09:00:00,Sell,62666.1,NaN,63292.761,62039.439000,62039.439000,63609.224805,1,2024-07-02 14:37:00,00:00:00,0.92931,100929.310100,,62666.1000,0.929310,2.664085,0
1,2024-07-04 09:00:00,Buy,57674.1,NaN,57097.359,57959.586795,58074.358254,56811.872205,1,2024-07-04 11:19:00,00:00:00,1.10894,102048.555834,,57385.7295,1.108940,2.664085,0
2,2024-07-05 09:00:00,Sell,NaN,NaN,NaN,NaN,NaN,NaN,Ignored,2024-07-04 11:19:00,None,0.00000,102048.555834,Signal around economic event,NaN,0.000000,2.664085,0
3,2024-07-06 11:00:00,Buy,56750.0,NaN,56182.500,57317.500000,57317.500000,55901.587500,1,2024-07-06 13:49:00,00:00:00,0.92931,102996.903370,,56750.0000,0.929310,2.664085,0
4,2024-07-07 01:00:00,Buy,57910.1,NaN,57330.999,58196.754995,58311.996094,57044.344005,1,2024-07-07 08:41:00,00:00:00,1.10894,104139.077477,,57620.5495,2.048556,2.664085,0
5,2024-07-07 09:00:00,Sell,57624.7,NaN,58200.947,57048.453000,57048.453000,58491.951735,1,2024-07-07 13:15:00,00:00:00,0.92931,105106.852442,,57624.7000,2.048556,2.664085,0
6,2024-07-07 13:00:00,Sell,NaN,NaN,NaN,NaN,NaN,NaN,Ignored,2024-07-07 13:15:00,None,0.00000,105106.852442,One open trade with the same side,NaN,2.048556,2.664085,0
7,2024-07-08 02:00:00,Buy,54930.0,NaN,NaN,55479.300000,55479.300000,54380.700000,1,2024-07-08 04:46:00,00:00:00,0.92931,106083.621038,,54930.0000,0.929310,2.664085,0
8,2024-07-09 06:00:00,Buy,57261.8,NaN,56689.182,57834.418000,57834.418000,56405.736090,1,2024-07-09 08:43:00,00:00:00,0.92931,107069.466843,,57261.8000,0.929310,2.664085,0
9,2024-07-10 01:00:00,Buy,NaN,NaN,NaN,NaN,NaN,NaN,Ignored,2024-07-09 08:43:00,None,0.00000,107069.466843,Signal around economic event,NaN,0.000000,2.664085,0


In [6]:
# Assuming the necessary variables are already defined, e.g., price_data, num_orders, etc.
signal_datetime = datetime(2024, 7, 29, 20, 0)
side = 'Buy'

# Call the updated handle_multiple_orders function
orders, average_entry_price, filled_orders, first_order_datetime, first_order_entry_price, tp_price_first_order, tp_price_second_order, sl_price = handle_multiple_orders(
    price_data,
    signal_datetime,
    side,
    num_orders,
    pct_orders,
    pct_order_range,
    entry_time_offset,
    percentage_change,
    open_order_elimination,
    tp_first_order,
    tp_second_order,
    sl
)

# Print the outputs
print(f"Orders: {orders}")
print(f"Average Entry Price: {average_entry_price}")
print(f"Filled Orders: {filled_orders}")
print(f"First Order Datetime: {first_order_datetime}")
print(f"First Order Entry Price: {first_order_entry_price}")
print(f"TP Price First Order: {tp_price_first_order}")
print(f"TP Price Second Order: {tp_price_second_order if filled_orders > 1 else 'N/A'}")
print(f"SL Price: {sl_price}")


Orders: [(Timestamp('2024-07-29 20:00:00'), 67276.7, 0.5, Timedelta('0 days 00:00:00')), (Timestamp('2024-07-29 23:17:00'), 66603.93299999999, 0.5, 197.0)]
Average Entry Price: 66940.31649999999
Filled Orders: 2
First Order Datetime: 2024-07-29 20:00:00
First Order Entry Price: 66603.93299999999
TP Price First Order: 67949.467
TP Price Second Order: 67743.60029799999
SL Price: 66270.91333499999


In [7]:
def calculate_metrics(group, initial_nav):
    total_trades = len(group[(group['Result'] == 1) | (group['Result'] == -1)])
    total_wins = len(group[group['Result'] == 1])
    total_losses = len(group[group['Result'] == -1])
    win_rate = total_wins / total_trades if total_trades > 0 else 0
    
    final_nav = group['NAV'].iloc[-1] if total_trades > 0 else initial_nav
    roi = ((final_nav - initial_nav) / initial_nav) * 100
    
    return {
        'Total Trades': total_trades,
        'Total Wins': total_wins,
        'Total Losses': total_losses,
        'Win Rate': win_rate,
        'ROI': roi,
        'NAV': final_nav
    }

def generate_report(price_data, signal_data, tp_sl_dict, output_directory):
    report_columns = ['Date', 'TP', 'SL', 'Total Trades', 'Total Wins', 'Total Losses', 'Win Rate', 'ROI', 'NAV']
    
    os.makedirs(output_directory, exist_ok=True)

    tp_values = next(iter(tp_sl_dict.values()))['tp']
    sl_values = next(iter(tp_sl_dict.values()))['sl']

    for tp, sl in zip(tp_values, sl_values):
        logging.info(f'Starting TP/SL combination: TP={tp}, SL={sl}')
        report_data = pd.DataFrame(columns=report_columns)
        
        for period, params in tp_sl_dict.items():
            logging.info(f'Processing period: {period}')
            
            # Filter the signal data for the current weekly period
            period_data = signal_data[signal_data['Datetime'].dt.to_period('W-MON') == period]
            if period_data.empty:
                logging.warning(f'No signal data for period: {period}')
                continue  # Skip to the next period
            
            # Backtest the trades for the current scenario
            trade_data = backtest_trades(price_data, period_data, tp, sl, entry_time_offset=0, 
                                         percentage_change=0.0005, open_order_elimination=120, ignore_time_interval_before=900, 
                                         ignore_time_interval_after=0)
            
            if trade_data.empty:
                logging.warning(f'No trades generated for period: {period}')
                continue  # Skip to the next period
            
            weekly_groups = trade_data.groupby(trade_data['Datetime'].dt.to_period('W-MON'))
            weekly_reports = []
            initial_nav = 100000
            for week, group in weekly_groups:
                logging.info(f'Calculating metrics for week: {week}')
                 # Get the start and end dates of the week
                week_start = week.start_time.strftime('%Y-%m-%d')
                week_end = (week.end_time - pd.Timedelta(days=1)).strftime('%Y-%m-%d')  # End date is one day before next week's start
                
                # Format the date as a range
                date_range = f'{week_start} to {week_end}'
                metrics = calculate_metrics(group, initial_nav)
                metrics['Date'] = date_range
                metrics['TP'] = tp
                metrics['SL'] = sl
                weekly_reports.append(pd.DataFrame([metrics]))
                initial_nav = metrics['NAV']

            if weekly_reports:
                weekly_report = pd.concat(weekly_reports, ignore_index=True)
                report_data = pd.concat([report_data, weekly_report], ignore_index=True)
        
        if not report_data.empty:
            now = datetime.now()        
            formatted_date = now.strftime('%Y-%m-%d')
            output_file_name = f'report_TP{tp}_SL{sl}_with_offset=0_pct=0.0005_ignore=900_parameters_{formatted_date}.csv'
            report_data.to_csv(os.path.join(output_directory, output_file_name), index=False)
            logging.info(f'Report saved: {output_file_name}')
        else:
            logging.warning(f'No data to save for TP={tp}, SL={sl}')
    
    logging.info('Report generation complete')
    return

In [8]:
# Define the start and end dates
start_date = pd.to_datetime('2024-01-01')
end_date = pd.to_datetime('2024-07-31')

# Generate a range of weekly periods including potentially unwanted periods
weekly_periods = pd.period_range(start=start_date, end=end_date, freq='W-MON')

# Filter out any period that starts before 2024-01-01
filtered_periods = [period for period in weekly_periods if period.start_time >= start_date]

# Create the tp_sl_dict with the specified tp and sl values for each filtered week
tp_sl_dict = {period: {'tp': [0.012], 'sl': [0.009]} for period in filtered_periods}

# Delete the specific period
period_to_delete = pd.Period('2024-07-30/2024-08-05', 'W-MON')
if period_to_delete in tp_sl_dict:
    del tp_sl_dict[period_to_delete]
    
tp_sl_dict    

{Period('2024-01-02/2024-01-08', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-01-09/2024-01-15', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-01-16/2024-01-22', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-01-23/2024-01-29', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-01-30/2024-02-05', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-02-06/2024-02-12', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-02-13/2024-02-19', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-02-20/2024-02-26', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-02-27/2024-03-04', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-03-05/2024-03-11', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-03-12/2024-03-18', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-03-19/2024-03-25', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-03-26/2024-04-01', 'W-MON'): {'tp': [0.012], 'sl': [0.009]},
 Period('2024-04-02/2024-

In [9]:
output_directory = 'E:\Signal Backtesting\Output\Backtest\weekly_backtest_2024_with_optimized_parameters_report'

# Generate the report
report_data = generate_report(price_data, signal_data, tp_sl_dict, output_directory)

TypeError: 'float' object cannot be interpreted as an integer